In [1]:
from openai import OpenAI
openai_client = OpenAI()
print("OpenAI client created")

OpenAI client created


In [2]:
# !uv add requests beautifulsoup4 scikit-learn
import requests
from bs4 import BeautifulSoup
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

#Azure Authentication
import msal

In [3]:
CLIENT_ID = "ab564a04-c563-4d89-be61-05c7883653dc"
AUTHORITY = "https://login.microsoftonline.com/consumers"
SCOPES = ["Notes.Read"]

app = msal.PublicClientApplication(
    client_id=CLIENT_ID,
    authority=AUTHORITY
)

In [4]:
flow = app.initiate_device_flow(
    scopes=SCOPES
)

# print(flow)
print(flow['message'])

To sign in, use a web browser to open the page https://www.microsoft.com/link and enter the code PK87M9XR to authenticate.


In [5]:
result = app.acquire_token_by_device_flow(flow)

if "access_token" in result:
    access_token = result["access_token"]
    print("Authentication successful!")
else:
    print("Authentication failed:")
    print(result.get("error"))
    print(result.get("error_description"))

Authentication successful!


In [6]:
import base64
import time
import requests

Now that we have the openai_client and access_token, we can now create functions to ingest the onenote before search

In [7]:
def image_bytes_to_data_url(image_bytes, mime_type):

    if not image_bytes:
        raise ValueError("image_bytes cannot be empty.")

    if not mime_type:
        mime_type = "image/png"

    encoded_image = base64.b64encode(
        image_bytes
    ).decode("utf-8")

    return (
        f"data:{mime_type};base64,"
        f"{encoded_image}"
    )

In [8]:
def clean_ocr_text(text):

    if not text:
        return ""

    text = text.strip()

    # Remove Markdown code fences that the model may add
    if text.startswith("```"):
        lines = text.splitlines()

        if lines and lines[0].strip().startswith("```"):
            lines = lines[1:]

        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]

        text = "\n".join(lines)

    return text.strip()

In [9]:
def extract_text_from_image(openai_client, image_bytes, mime_type):

    if not image_bytes:
        return ""

    image_data_url = image_bytes_to_data_url(image_bytes, mime_type)

    response = openai_client.responses.create(
        model="gpt-4o-mini",
        input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": (
                            "Extract all readable text from this image. "
                            "Preserve the original wording, numbers, dates, "
                            "error codes, labels, headings, table values, "
                            "and important line structure as accurately as possible. "
                            "Do not explain, summarize, or interpret the image. "
                            "Do not invent text that is not visible. "
                            "Return only the extracted text. "
                            "Do not wrap the output in Markdown code fences. "
                            "If there is no readable text, return an empty string."
                        )
                    },
                    {
                        "type": "input_image",
                        "image_url": image_data_url
                    }
                ]
            }
        ]
    )

    return clean_ocr_text(response.output_text)

In [10]:
def graph_get(url, headers, max_retries=5, timeout=30):

    for attempt in range(max_retries + 1):

        response = requests.get(url, headers=headers, timeout=timeout)

        # Request succeeded
        if response.ok:
            return response

        # Microsoft Graph is throttling us
        if response.status_code == 429:

            # Graph may tell us how many seconds to wait
            retry_after = response.headers.get(
                "Retry-After"
            )

            if retry_after is not None:
                wait_seconds = int(retry_after)

            else:
                # Fallback exponential backoff:
                # 1, 2, 4, 8, 16...
                wait_seconds = 2 ** attempt

            print(
                f"Microsoft Graph throttled the request. "
                f"Waiting {wait_seconds} seconds before retrying..."
            )

            time.sleep(wait_seconds)

            continue

        # Some server errors may be temporary
        if response.status_code in {
            500,
            502,
            503,
            504
        }:

            if attempt < max_retries:

                wait_seconds = 2 ** attempt

                print(
                    f"Temporary Microsoft Graph error "
                    f"({response.status_code}). "
                    f"Retrying in {wait_seconds} seconds..."
                )

                time.sleep(wait_seconds)

                continue

        # Other errors should be raised normally
        response.raise_for_status()


    raise RuntimeError(
        f"Microsoft Graph request failed after "
        f"{max_retries + 1} attempts: {url}"
    )

In [11]:
def read_onenote(access_token, notebook_name, openai_client):

    headers = {
        "Authorization": f"Bearer {access_token}"
    }

    base_url = "https://graph.microsoft.com/v1.0/me/onenote"

    documents = []


    # ---------------------------------------------------------
    # 1. GET ALL NOTEBOOKS
    # ---------------------------------------------------------

    notebooks_response = graph_get(
        f"{base_url}/notebooks",
        headers
    )

    notebooks = notebooks_response.json()["value"]


    # ---------------------------------------------------------
    # 2. FIND THE REQUESTED NOTEBOOK
    # ---------------------------------------------------------

    notebook = None

    for item in notebooks:

        if item["displayName"] == notebook_name:

            notebook = item
            break


    if notebook is None:

        raise ValueError(
            f"Notebook '{notebook_name}' was not found."
        )


    notebook_id = notebook["id"]


    # ---------------------------------------------------------
    # 3. GET SECTIONS IN THE NOTEBOOK
    # ---------------------------------------------------------

    sections_response = graph_get(
        f"{base_url}/notebooks/{notebook_id}/sections",
        headers
    )

    sections = sections_response.json()["value"]


    # ---------------------------------------------------------
    # 4. PROCESS EACH SECTION
    # ---------------------------------------------------------

    for section in sections:

        section_id = section["id"]
        section_name = section["displayName"]


        # -----------------------------------------------------
        # 5. GET PAGES IN THIS SECTION
        # -----------------------------------------------------

        pages_response = graph_get(
            f"{base_url}/sections/{section_id}/pages",
            headers
        )

        pages = pages_response.json()["value"]


        # -----------------------------------------------------
        # 6. PROCESS EACH PAGE
        # -----------------------------------------------------

        for page in pages:

            page_id = page["id"]
            page_title = page["title"]


            # -------------------------------------------------
            # 7. GET THE PAGE CONTENT
            # -------------------------------------------------

            content_response = graph_get(
                f"{base_url}/pages/{page_id}/content",
                headers
            )

            page_html = content_response.text


            # -------------------------------------------------
            # 8. PARSE THE ONENOTE HTML
            # -------------------------------------------------

            soup = BeautifulSoup(
                page_html,
                "html.parser"
            )


            # -------------------------------------------------
            # 9. EXTRACT PAGE TEXT
            # -------------------------------------------------

            page_text = soup.get_text(
                separator=" ",
                strip=True
            )


            # -------------------------------------------------
            # 10. FIND ALL IMAGES ON THE PAGE
            # -------------------------------------------------

            image_elements = soup.find_all("img")

            images = []


            # -------------------------------------------------
            # 11. PROCESS EACH IMAGE
            # -------------------------------------------------

            for image_number, image_element in enumerate(
                image_elements,
                start=1
            ):

                # ---------------------------------------------
                # IMAGE RESOURCE URLS
                # ---------------------------------------------

                image_url = image_element.get("src")

                fullres_url = image_element.get("data-fullres-src")

                # Prefer the original/high-resolution image
                image_resource_url = (
                    fullres_url
                    or image_url
                )


                # ---------------------------------------------
                # 12. EXTRACT IMAGE METADATA
                # ---------------------------------------------

                image_metadata = {

                    "image_number": image_number,

                    "alt_text": image_element.get(
                        "alt"
                    ),

                    "width": image_element.get(
                        "width"
                    ),

                    "height": image_element.get(
                        "height"
                    ),

                    "image_type": (
                        image_element.get(
                            "data-fullres-src-type"
                        )
                        or
                        image_element.get(
                            "data-src-type"
                        )
                        or
                        "image/png"
                    ),

                    "data_id": image_element.get(
                        "data-id"
                    ),

                    "data_index": image_element.get(
                        "data-index"
                    ),

                    "data_options": image_element.get(
                        "data-options"
                    ),

                    "original_source_url":
                        image_element.get(
                            "data-render-original-src"
                        )
                }


                # ---------------------------------------------
                # 13. EXTRACT TEXT AROUND THE IMAGE
                # ---------------------------------------------

                parent = image_element.parent
                surrounding_text = ""

                if parent:
                    surrounding_text = parent.get_text(
                        separator=" ",
                        strip=True
                    )


                # ---------------------------------------------
                # 14. DOWNLOAD THE IMAGE
                # ---------------------------------------------

                image_bytes = None
                image_download_error = None


                if image_resource_url:

                    try:

                        image_response = requests.get(
                            image_resource_url,
                            headers=headers
                        )

                        image_response.raise_for_status()

                        image_bytes = (
                            image_response.content
                        )


                    except requests.RequestException as error:

                        image_download_error = str(
                            error
                        )


                else:

                    image_download_error = (
                        "No image resource URL was available."
                    )


                # ---------------------------------------------
                # 15. PERFORM OCR
                # ---------------------------------------------

                ocr_text = ""
                ocr_error = None


                if image_bytes:

                    try:

                        ocr_text = extract_text_from_image(
                            openai_client,
                            image_bytes,
                            image_metadata["image_type"]
                        )


                    except Exception as error:

                        ocr_error = str(
                            error
                        )


                # ---------------------------------------------
                # 16. CREATE NORMALIZED IMAGE DOCUMENT
                # ---------------------------------------------

                image_document = {

                    "id": (
                        image_metadata["data_id"]
                        or
                        f"{page_id}-image-{image_number}"
                    ),

                    "type": "image",

                    "source_type": "onenote_image",

                    "parent_page_id": page_id,

                    "notebook": notebook_name,

                    "section": section_name,

                    "page_title": page_title,

                    # Resource references
                    "image_url": image_url,

                    "fullres_url": fullres_url,

                    # Metadata
                    "metadata": image_metadata,

                    # Retrieval information
                    "surrounding_text": surrounding_text,

                    "ocr_text": ocr_text,

                    # Processing information
                    "download_error":
                        image_download_error,

                    "ocr_error":
                        ocr_error
                }


                images.append(
                    image_document
                )


            # -------------------------------------------------
            # 17. CREATE NORMALIZED PAGE DOCUMENT
            # -------------------------------------------------

            document = {

                "id": page_id,

                "type": "page",

                "source_type": "onenote",

                "notebook": notebook_name,

                "section": section_name,

                "title": page_title,

                "content": page_text,

                "last_modified": page.get(
                    "lastModifiedDateTime"
                ),

                "images": images
            }


            # -------------------------------------------------
            # 18. ADD PAGE TO DOCUMENT COLLECTION
            # -------------------------------------------------

            documents.append(
                document
            )


    # ---------------------------------------------------------
    # 19. RETURN NORMALIZED ONENOTE DOCUMENTS
    # ---------------------------------------------------------

    return documents

In [12]:
documents = read_onenote(
    access_token=access_token,
    notebook_name="testnote",
    openai_client=openai_client
)

#print(documents)

Microsoft Graph throttled the request. Waiting 5 seconds before retrying...


In [13]:
documents[2]

{'id': '0-ebbf72f227f646c89334a7f9efbde27c!1-3EFF8F91DE152C4E!sda6fbd34bc1b42a7b0a6c34a928393ff',
 'type': 'page',
 'source_type': 'onenote',
 'notebook': 'testnote',
 'section': 'Desktop Support',
 'title': 'Maintaining Your Windows Desktop',
 'content': 'Maintaining Your Windows Desktop Regular Maintenance Restart the computer regularly so operating-system updates can complete. Employees should avoid leaving university computers running for long periods without restarting them. Windows Update should be allowed to install approved security and system updates. Storage Users should periodically remove unnecessary downloads and temporary files. Important university documents should be stored in approved university storage locations rather than only on the computer\'s local drive. Employees should maintain sufficient free disk space for Windows updates and normal application operation. Performance Problems If the computer becomes unusually slow: Restart the computer. Check whether Windows

In [14]:
documents[4]

{'id': '0-0114fb6567524a8589a7ff47beed350d!1-3EFF8F91DE152C4E!s72c14ae75a9b478a857357c062b481d0',
 'type': 'page',
 'source_type': 'onenote',
 'notebook': 'testnote',
 'section': 'IT Policies',
 'title': 'Acceptable Use Policy',
 'content': 'Acceptable Use Policy Purpose University computers and technology resources are provided to support teaching, research, administration, and other approved university activities. User Responsibilities Employees should use university computers primarily for authorized university business. Users must protect confidential university information and should not share passwords or authentication credentials with other individuals. Employees should lock their computers whenever they leave their workstation unattended. On Windows computers, users can press Windows + L to immediately lock the computer. Software Installation Employees should not install unauthorized software on university-owned computers. Software required for university work should be obtain

Now that we have ingested the onenote data and gotten the documents, we can now build the search engine! YAY!

In [15]:
from copy import deepcopy

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [16]:
class OneNoteSearchEngine:

    def __init__(self, documents):

        if not isinstance(documents, list) or not documents:
            raise ValueError(
                "documents must be a non-empty list."
            ) #Or just return ""

        self.documents = documents

        self.searchable_text = [
            self._build_searchable_text(document)
            for document in documents
        ]

        self.vectorizer = TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            strip_accents="unicode",
            ngram_range=(1, 2)
        )

        try:
            self.document_matrix = self.vectorizer.fit_transform(
                self.searchable_text
            )

        except ValueError as error:
            raise ValueError(
                "The documents do not contain enough searchable text "
                "to build the search index."
            ) from error


    def _build_searchable_text(self, document):

        text_parts = [
            document.get("notebook", ""),
            document.get("section", ""),
            document.get("title", ""),
            document.get("content", "")
        ]

        for image in document.get("images", []):

            text_parts.extend([
                image.get("surrounding_text", ""),
                image.get("ocr_text", "")
            ])

        return "\n".join(
            text.strip()
            for text in text_parts
            if isinstance(text, str) and text.strip()
        )


    def search(
        self,
        query,
        num_results=5,
        min_score=0.0
    ):

        if not isinstance(query, str) or not query.strip():
            raise ValueError(
                "query must be a non-empty string."
            )

        if not isinstance(num_results, int) or num_results <= 0:
            raise ValueError(
                "num_results must be a positive integer."
            )

        if not isinstance(min_score, (int, float)):
            raise ValueError(
                "min_score must be a number."
            )

        if not 0.0 <= min_score <= 1.0:
            raise ValueError(
                "min_score must be between 0.0 and 1.0."
            )

        query = query.strip()

        query_vector = self.vectorizer.transform(
            [query]
        )

        scores = cosine_similarity(
            self.document_matrix,
            query_vector
        ).ravel()

        ranked_indexes = scores.argsort()[::-1]

        results = []

        for index in ranked_indexes:

            score = float(scores[index])

            if score <= min_score:
                continue

            result = deepcopy(
                self.documents[index]
            )

            result["score"] = score

            results.append(result)

            if len(results) >= num_results:
                break

        return results

In [17]:
search_engine = OneNoteSearchEngine(
    documents
)

In [19]:
results = search_engine.search(
    "Which faculty member had a Dell Latitude 5420 with low disk space?",
    num_results=3
)

for position, result in enumerate(results, start=1):

    print("=" * 80)
    print(f"RESULT {position}")
    print("=" * 80)

    print(f"Notebook : {result['notebook']}")
    print(f"Section  : {result['section']}")
    print(f"Title    : {result['title']}")
    print(f"Score    : {result['score']:.3f}")
    print()

RESULT 1
Notebook : testnote
Section  : Desktop Support
Title    : Maintaining Your Windows Desktop
Score    : 0.317

RESULT 2
Notebook : testnote
Section  : IT Policies
Title    : Acceptable Use Policy
Score    : 0.004



In [20]:
for image in results[0]["images"]:

    print("=" * 80)
    print(f"IMAGE {image['metadata']['image_number']}")
    print("=" * 80)

    print("Surrounding text:")
    print(image["surrounding_text"])

    print()
    print("OCR text:")
    print(image["ocr_text"])

    print()

IMAGE 1
Surrounding text:
About Regular Maintenance & Storage and performance

OCR text:
Regular Maintenance  
Keep your computer updated and running smoothly.  

Restart Regularly  
Restart your computer regularly so operating-system updates can complete. Avoid leaving your computer running for long periods without restarting.  

Windows Update  
Allow Windows Update to install approved security and system updates.  

Keep Security Enabled  
Do not disable antivirus software, endpoint protection, firewalls, or other security controls installed by IT.  

Storage  
Remove unnecessary downloads and temporary files periodically.  
Store important university documents in approved university storage locations (e.g., network drives, OneDrive, SharePoint).  
Maintain sufficient free disk space for Windows updates and normal application operation.  

If Your Computer Becomes Slow  
1. Restart the computer.  
2. Check whether Windows Update is currently installing updates.  
3. Close applicatio

In [32]:
#use this
class OneNoteSearchEngines:

    def __init__(self, documents):

        # Validate documents
        if not isinstance(documents, list) or not documents:
            raise ValueError(
                "documents must be a non-empty list."
            )

        self.documents = documents

        # Build searchable text for every document
        self.searchable_text = []
        for document in documents:
            dtext = self._build_searchable_text(document)
            self.searchable_text.append(dtext)

        # self.searchable_text = [
        #     self._build_searchable_text(document)
        #     for document in documents
        # ]

        # Create the TF-IDF vectorizer
        self.vectorizer = TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            strip_accents="unicode",
            ngram_range=(1, 2)
        )

        # Build the TF-IDF document index once
        try:
            self.document_matrix = self.vectorizer.fit_transform(
                self.searchable_text
            )

        except ValueError as error:
            raise ValueError(
                "The documents do not contain enough searchable text "
                "to build the search index."
            ) from error


    def _build_searchable_text(self, document):

        # Start with normal OneNote page information
        text_parts = [
            document.get("notebook", ""),
            document.get("section", ""),
            document.get("title", ""),
            document.get("content", "")
        ]

        # Add searchable information extracted from images
        for image in document.get("images", []):

            text_parts.extend([
                image.get("surrounding_text", ""),
                image.get("ocr_text", "")
            ])

        # Remove empty values and combine everything
        return "\n".join(
            text.strip()
            for text in text_parts
            if isinstance(text, str) and text.strip()
        )


    def search(
        self,
        query,
        num_results=5,
        min_score=0.0
    ):

        # Validate query
        if not isinstance(query, str) or not query.strip():
            raise ValueError(
                "query must be a non-empty string."
            )

        # Validate requested number of results
        if not isinstance(num_results, int) or num_results <= 0:
            raise ValueError(
                "num_results must be a positive integer."
            )

        # Validate minimum similarity score
        if not isinstance(min_score, (int, float)):
            raise ValueError(
                "min_score must be a number."
            )

        if not 0.0 <= min_score <= 1.0:
            raise ValueError(
                "min_score must be between 0.0 and 1.0."
            )

        query = query.strip()

        # Convert query using the SAME fitted vectorizer
        query_vector = self.vectorizer.transform(
            [query]
        )

        # Compare the query with every indexed document
        scores = cosine_similarity(
            self.document_matrix,
            query_vector
        ).ravel()

        # Rank documents from highest to lowest similarity
        ranked_indexes = scores.argsort()[::-1]

        results = []

        for index in ranked_indexes:

            score = float(scores[index])

            # Ignore results that do not meet the threshold
            if score <= min_score:
                continue

            # Copy the complete original document
            result = deepcopy(
                self.documents[index]
            )

            # Add this query's similarity score
            result["score"] = score

            results.append(result)

            # Stop after collecting enough results
            if len(results) >= num_results:
                break

        return results

In [26]:
search_engine2 = OneNoteSearchEngines(
    documents
)

In [27]:
results = search_engine2.search(
    "J. Rivera Library Room 218 Dell Latitude 5420",
    num_results=3
)

for result in results:
    print(result["title"], result["score"])

Maintaining Your Windows Desktop 0.0827552823467772


In [30]:
results = search_engine2.search(
    "Which technician handled the service note in Library Room 218?",
    num_results=3
)

for result in results:
    print(result["title"], result["score"])

Maintaining Your Windows Desktop 0.05380172708701142
Acceptable Use Policy 0.0050918005237795
